In [31]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import joblib
import time
from tensorflow.keras.models import load_model
from scipy.optimize import minimize
from scipy.optimize import differential_evolution
import pyswarms as ps
from scipy.optimize import dual_annealing
from scipy.optimize import dual_annealing
from skopt import gp_minimize
from skopt.space import Real
from skopt import gp_minimize
from skopt.space import Real
import warnings
warnings.filterwarnings("ignore")


In [32]:
# L-BFGS-B
model = load_model('../artifacts/mlp_model.h5')
scaler = joblib.load('../artifacts/scaler.pkl')

def objective(inputs):
    inputs_scaled = scaler.transform([inputs])
    predicted = model.predict(inputs_scaled, verbose=0)[0][0]
    return predicted  # single scalar only

bounds = [
    (0.000755, 0.000817),
    (-2.302585, 2.708050),
    (0.1, 0.7)
]

x0 = [0.000786, 0.2, 0.4]

result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds)

print("Optimal Inputs Found:")
print(f"  1/T            : {result.x[0]:.6f}")
print(f"  ln strain rate : {result.x[1]:.6f}")
print(f"  Strain         : {result.x[2]:.6f}")
print(f"  Predicted σ/σmax: {result.fun:.4f}")

# Conversion to engineering units
optimal_inv_T = result.x[0]
optimal_ln_sr = result.x[1]
optimal_strain = result.x[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {result.fun:.4f}")
print("=" * 45)


2026-07-11 07:22:51,231 - absl - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Optimal Inputs Found:
  1/T            : 0.000755
  ln strain rate : 0.200000
  Strain         : 0.400000
  Predicted σ/σmax: 0.5333
  OPTIMAL PROCESS PARAMETERS — AISI 304
  Temperature   : 1051.4 °C
  Strain Rate   : 1.2214 s⁻¹
  Strain        : 0.40
  Min σ/σmax    : 0.5333


In [33]:
xgb_model = joblib.load('../artifacts/xgb_model.pkl')
scaler = joblib.load('../artifacts/scaler.pkl')

def objective(inputs):
    inputs_scaled = scaler.transform([inputs])
    return xgb_model.predict(inputs_scaled)[0]

# Bounds: [1/T, ln_strain_rate, Strain]
bounds = [
    (0.000755, 0.000817),
    (-2.302585, 2.708050),
    (0.1, 0.7)
]

In [34]:
# Differential Evolution
start = time.time()
de_result = differential_evolution(
    objective, bounds,
    strategy='best1bin',
    maxiter=200,
    popsize=20,
    tol=1e-7,
    seed=42
)
de_time = time.time() - start

print("Differential Evolution — Optimal Inputs Found:")
print(f"  1/T            : {de_result.x[0]:.6f}")
print(f"  ln strain rate : {de_result.x[1]:.6f}")
print(f"  Strain         : {de_result.x[2]:.6f}")
print(f"  Predicted σ/σmax: {de_result.fun:.4f}")
print(f"  Time           : {de_time:.2f}s")


# Conversion to engineering units
optimal_inv_T = de_result.x[0]
optimal_ln_sr = de_result.x[1]
optimal_strain = de_result.x[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {de_result.fun:.4f}")
print("=" * 45)


Differential Evolution — Optimal Inputs Found:
  1/T            : 0.000762
  ln strain rate : -2.089814
  Strain         : 0.100289
  Predicted σ/σmax: 0.1755
  Time           : 1.68s
  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304
  Temperature   : 1039.2 °C
  Strain Rate   : 0.1237 s⁻¹
  Strain        : 0.10
  Min σ/σmax    : 0.1755


In [35]:
#  Particle Swarm Optimization
def objective_pso(x):
    # pyswarms passes a 2D array of shape (n_particles, n_dims)
    return np.array([objective(row) for row in x])

lb = np.array([b[0] for b in bounds])
ub = np.array([b[1] for b in bounds])

options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
optimizer = ps.single.GlobalBestPSO(
    n_particles=30, dimensions=3,
    options=options, bounds=(lb, ub)
)

start = time.time()
pso_cost, pso_pos = optimizer.optimize(objective_pso, iters=200, verbose=False)
pso_time = time.time() - start

print("Particle Swarm Optimization — Optimal Inputs Found:")
print(f"  1/T            : {pso_pos[0]:.6f}")
print(f"  ln strain rate : {pso_pos[1]:.6f}")
print(f"  Strain         : {pso_pos[2]:.6f}")
print(f"  Predicted σ/σmax: {pso_cost:.4f}")
print(f"  Time           : {pso_time:.2f}s")


# Conversion to engineering units
optimal_inv_T = pso_pos[0]
optimal_ln_sr = pso_pos[1]
optimal_strain = pso_pos[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS (Particle Swarm Optimization — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {pso_cost:.4f}")
print("=" * 45)


Particle Swarm Optimization — Optimal Inputs Found:
  1/T            : 0.000783
  ln strain rate : -0.128982
  Strain         : 0.105878
  Predicted σ/σmax: 0.1755
  Time           : 6.23s
  OPTIMAL PROCESS PARAMETERS (Particle Swarm Optimization — AISI 304
  Temperature   : 1003.4 °C
  Strain Rate   : 0.8790 s⁻¹
  Strain        : 0.11
  Min σ/σmax    : 0.1755


In [36]:
# Simulated Annealing
start = time.time()
sa_result = dual_annealing(objective, bounds, seed=42, maxiter=200)
sa_time = time.time() - start

print("Simulated Annealing — Optimal Inputs Found:")
print(f"  1/T            : {sa_result.x[0]:.6f}")
print(f"  ln strain rate : {sa_result.x[1]:.6f}")
print(f"  Strain         : {sa_result.x[2]:.6f}")
print(f"  Predicted σ/σmax: {sa_result.fun:.4f}")
print(f"  Time           : {sa_time:.2f}s")

# Conversion to engineering units
optimal_inv_T = sa_result.x[0]
optimal_ln_sr = sa_result.x[1]
optimal_strain = sa_result.x[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {sa_result.fun:.4f}")
print("=" * 45)



Simulated Annealing — Optimal Inputs Found:
  1/T            : 0.000785
  ln strain rate : -1.554659
  Strain         : 0.113296
  Predicted σ/σmax: 0.1755
  Time           : 1.18s
  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304
  Temperature   : 1000.3 °C
  Strain Rate   : 0.2113 s⁻¹
  Strain        : 0.11
  Min σ/σmax    : 0.1755


In [37]:
#  Bayesian Optimization
space = [
    Real(bounds[0][0], bounds[0][1], name='inv_T'),
    Real(bounds[1][0], bounds[1][1], name='ln_strain_rate'),
    Real(bounds[2][0], bounds[2][1], name='strain')
]

start = time.time()
bo_result = gp_minimize(
    objective, space,
    n_calls=50, n_initial_points=10,
    random_state=42
)
bo_time = time.time() - start

print("Bayesian Optimization — Optimal Inputs Found:")
print(f"  1/T            : {bo_result.x[0]:.6f}")
print(f"  ln strain rate : {bo_result.x[1]:.6f}")
print(f"  Strain         : {bo_result.x[2]:.6f}")
print(f"  Predicted σ/σmax: {bo_result.fun:.4f}")
print(f"  Time           : {bo_time:.2f}s")

# Conversion to engineering units
optimal_inv_T = bo_result.x[0]
optimal_ln_sr = bo_result.x[1]
optimal_strain = bo_result.x[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {bo_result.fun:.4f}")
print("=" * 45)

Bayesian Optimization — Optimal Inputs Found:
  1/T            : 0.000781
  ln strain rate : -0.730264
  Strain         : 0.121169
  Predicted σ/σmax: 0.1755
  Time           : 18.24s
  OPTIMAL PROCESS PARAMETERS (DIfferential Evolution — AISI 304
  Temperature   : 1006.9 °C
  Strain Rate   : 0.4818 s⁻¹
  Strain        : 0.12
  Min σ/σmax    : 0.1755


In [38]:

comparison = pd.DataFrame([
    {'Algorithm': 'Differential Evolution', 'sigma_ratio': de_result.fun, 'Time (s)': de_time},
    {'Algorithm': 'Particle Swarm',         'sigma_ratio': pso_cost,      'Time (s)': pso_time},
    {'Algorithm': 'Simulated Annealing',    'sigma_ratio': sa_result.fun, 'Time (s)': sa_time},
    {'Algorithm': 'Bayesian Optimization',  'sigma_ratio': bo_result.fun, 'Time (s)': bo_time},
])
comparison = comparison.sort_values('sigma_ratio').reset_index(drop=True)
print(comparison.to_string(index=False))

             Algorithm  sigma_ratio  Time (s)
Differential Evolution     0.175476  1.678644
        Particle Swarm     0.175476  6.230162
   Simulated Annealing     0.175476  1.179618
 Bayesian Optimization     0.175476 18.240880


In [40]:
def to_engineering_units(x):
    inv_T, ln_sr, strain = x
    temp_c = (1 / inv_T) - 273.15
    strain_rate = np.exp(ln_sr)
    return temp_c, strain_rate, strain

# name, x (optimal inputs), fun (σ/σmax), time, surrogate model used
algorithms_results = [
    ('L-BFGS-B',              result.x,    result.fun,    None,    'ANN'),
    ('Differential Evolution', de_result.x, de_result.fun, de_time, 'XGBoost'),
    ('Particle Swarm',         pso_pos,     pso_cost,      pso_time,'XGBoost'),
    ('Simulated Annealing',    sa_result.x, sa_result.fun, sa_time, 'XGBoost'),
    ('Bayesian Optimization',  bo_result.x, bo_result.fun, bo_time, 'XGBoost'),
]

rows = []
for name, x, fun, t, surrogate in algorithms_results:
    temp_c, strain_rate, strain = to_engineering_units(x)
    rows.append({
        'Algorithm': name,
        'Surrogate': surrogate,
        'Temperature (°C)': round(temp_c, 1),
        'Strain Rate (s⁻¹)': round(strain_rate, 4),
        'Strain': round(strain, 3),
        'σ/σmax': round(float(fun), 4),
        'Time (s)': round(t, 2) if t is not None else '—'
    })

comparison = pd.DataFrame(rows)
comparison = comparison.sort_values('σ/σmax').reset_index(drop=True)
print(comparison.to_string(index=False))

             Algorithm Surrogate  Temperature (°C)  Strain Rate (s⁻¹)  Strain  σ/σmax Time (s)
Differential Evolution   XGBoost            1039.2             0.1237   0.100  0.1755     1.68
        Particle Swarm   XGBoost            1003.4             0.8790   0.106  0.1755     6.23
   Simulated Annealing   XGBoost            1000.3             0.2113   0.113  0.1755     1.18
 Bayesian Optimization   XGBoost            1006.9             0.4818   0.121  0.1755    18.24
              L-BFGS-B       ANN            1051.4             1.2214   0.400  0.5333        —


## Optimization Conclusions

The trained MLP was used as a differentiable surrogate model to
determine the optimal hot deformation parameters for AISI 316
stainless steel. Optimization was performed using the L-BFGS-B
gradient-based algorithm via scipy.optimize.minimize, which
navigates the smooth ANN prediction surface to find the input
combination that minimizes normalized flow stress.

**Why Minimize Flow Stress**
In hot deformation processing, minimizing flow stress is the
primary engineering objective. Lower flow stress corresponds to
conditions where the material offers least resistance to
deformation, reducing the energy and force requirements of the
forming process, minimizing tool wear, and improving the overall
efficiency of hot working operations.

**Why ANN as Surrogate**
Tree-based models such as XGBoost and Random Forest produce
discontinuous, step-like prediction surfaces with no mathematical
gradient. Gradient-based optimization requires a smooth,
differentiable function. The ANN, built entirely from differentiable
operations, uniquely satisfies this requirement, making it the
only model among those evaluated that supports this optimization
approach.

**Optimization Bounds**
The search space was constrained to the physical range of the
experimental data:

| Parameter     | Lower Bound      | Upper Bound      |
|---------------|------------------|------------------|
| Temperature   | 0.000755 K⁻¹     | 0.000817 K⁻¹     |
| ln Strain Rate| -2.3026          | 2.7081           |
| Strain        | 0.1              | 0.7              |

**Optimal Parameters Found**

| Parameter     | Optimal Value    |
|---------------|------------------|
| Temperature   | 1051.4 °C        |
| Strain Rate   | 1.2214 s⁻¹       |
| Strain        | 0.40             |
| Min σ/σmax    | 0.5341           |

**Interpretation**
The optimization converged to a deformation temperature of 1051.4°C,
a strain rate of 1.2214 s⁻¹, and a strain of 0.40. At these
conditions the model predicts a normalized flow stress of 0.5341,
the lowest achievable value within the physical bounds of the
experimental dataset. These represent the most energy-efficient
hot working parameters for AISI 304 stainless steel within the
tested operating range.

The result is physically consistent, higher temperature reduces
flow stress by increasing atomic mobility and promoting dynamic
recovery, while the intermediate strain rate avoids the stress
hardening associated with very high strain rates.